# 🛡️ Pipeline Hybrid NIDS: Stacked Autoencoder (SAE) + Cascaded Isolation Forest
## Optimasi Pendeteksian Anomali & Penurunan False Negative Rate (FNR)

Notebook ini mengimplementasikan pipeline hybrid end-to-end untuk mendeteksi intrusi jaringan (NIDS), khususnya serangan zero-day / unseen attack (seperti DDoS).

### 🎯 Tujuan Utama
Menurunkan **False Negative Rate (FNR)** (yaitu meminimalkan serangan yang terlewat / lolos) tanpa mengorbankan False Positive Rate (FPR) secara berlebihan melalui:
1. **Cascaded Feature Fusion**: Melatih Isolation Forest pada representasi laten SAE ($Z$) yang diperkaya dengan residual *Reconstruction Error* (MSE).
2. **Disjunctive (OR) Decision Fusion**: Menggabungkan keputusan $\text{Pred}_{SAE} \lor \text{Pred}_{IF}$ yang secara matematis menjamin $\text{FN}_{hybrid} \le \min(\text{FN}_{SAE}, \text{FN}_{IF})$ sehingga $\text{FNR}$ terbukti menurun.
3. **Modular Codebase**: Memanfaatkan modul `src/` yang modular, bersih, dan reproducible untuk penulisan skripsi.

### 1. Inisialisasi dan Konfigurasi Pipeline

In [1]:
import sys
from pathlib import Path

# Tambahkan root project ke sys.path
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import PipelineConfig
from src.data_loader import (
    load_partition_metadata,
    load_scaler_params,
    load_evaluation_dataset,
)
from src.sae_model import load_sae, train_sae, save_sae, StackedAutoencoder
from src.iforest_model import CascadedIForest
from src.hybrid_fusion import HybridDetector
from src.evaluation import (
    evaluate_pipeline_models,
    plot_roc_pr_curves,
    plot_confusion_matrices,
)

print("Modul src berhasil dimuat!")

Modul src berhasil dimuat!


In [2]:
# Inisialisasi konfigurasi pipeline
config = PipelineConfig()
config.base_dir = ROOT_DIR
config.resolve_paths()

# Set mode feature IForest dan strategi fusi
config.iforest_feature_mode = "latent_mse"  # [Z, MSE]
config.fusion_strategy = "or_rule"          # Disjunctive OR untuk minimasi FNR
config.threshold_percentile = 95.0          # 95th percentile benign (FPR target ~5%)

meta = load_partition_metadata(config.metadata_path)
feature_columns = meta["feature_columns"]
mins, maxs = load_scaler_params(config.scaler_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Metadata Dataset   :", config.metadata_path)
print("Jumlah Fitur       :", len(feature_columns))
print("Device PyTorch     :", device)
print("Feature Mode IF    :", config.iforest_feature_mode)
print("Fusion Strategy    :", config.fusion_strategy)

Metadata Dataset   : I:\skripsi\NIDS\notebooks\prepared_data\partition_metadata.json
Jumlah Fitur       : 39
Device PyTorch     : cpu
Feature Mode IF    : latent_mse
Fusion Strategy    : or_rule


### 2. Memuat / Melatih Stacked Autoencoder (SAE)

Stacked Autoencoder bertugas mempelajari representasi manifold data lalu lintas normal (*benign*). Serangan yang menyimpang dari pola normal akan menghasilkan nilai *Reconstruction Error* (MSE) yang tinggi.

In [3]:
train_files = sorted(config.train_dir.glob("*.parquet"))
print(f"Ditemukan {len(train_files)} file partisi training benign.")

# Opsi: Set True jika ingin melatih ulang SAE dari nol
RETRAIN_SAE = False

if RETRAIN_SAE or not config.sae_weights_path.exists():
    print("Melatih Stacked Autoencoder...")
    sae_model = StackedAutoencoder(
        input_dim=len(feature_columns),
        encoder_dims=config.encoder_dims,
        latent_dim=config.latent_dim,
        decoder_dims=config.decoder_dims,
        output_activation=config.output_activation,
    )
    history = train_sae(
        model=sae_model,
        train_files=train_files,
        feature_columns=feature_columns,
        mins=mins,
        maxs=maxs,
        epochs=config.epochs,
        batch_size=config.batch_size,
        learning_rate=config.learning_rate,
        device=device,
    )
    save_sae(
        model=sae_model,
        weights_path=config.sae_weights_path,
        config_path=config.sae_config_path,
        history=history,
        history_path=config.artifacts_dir / "sae_training_history.json",
    )
    print("Pelatihan SAE selesai dan bobot berhasil disimpan.")
else:
    print(f"Memuat model SAE yang sudah dilatih dari: {config.sae_weights_path}")
    sae_model = load_sae(
        config.sae_weights_path,
        config.sae_config_path,
        device=device,
    )
    print("Model SAE siap digunakan!")

Ditemukan 5 file partisi training benign.
Memuat model SAE yang sudah dilatih dari: I:\skripsi\NIDS\notebooks\artifacts\sae_autoencoder.pth
Model SAE siap digunakan!


### 3. Cascaded Feature Extraction & Training Isolation Forest

**Mengapa Cascaded Feature Fusion $[Z, \text{MSE}]$?**
- Jika Isolation Forest hanya dilatih pada vektor laten $Z$, ia kehilangan informasi esensial tentang *apa yang gagal direkonstruksi* oleh SAE.
- Dengan menggabungkan vektor laten $Z$ (dimensi 16) dengan *Reconstruction Error* MSE (dimensi 1), Isolation Forest memiliki akses ke topologi representasi terkompresi sekaligus besaran deviasi rekonstruksi.

In [4]:
print("Mengambil sampel data train benign untuk melatih Cascaded IForest...")
X_train_sample, _, _ = load_evaluation_dataset(
    files=train_files,
    feature_columns=feature_columns,
    mins=mins,
    maxs=maxs,
    max_samples=config.max_train_samples_for_iforest,
    seed=config.seed,
)
print(f"Total sampel train: {len(X_train_sample):,}")

# Pisahkan subset fit dan kalibrasi
calib_size = int(len(X_train_sample) * config.iforest_calibration_ratio)
X_fit = X_train_sample[calib_size:]
X_calib = X_train_sample[:calib_size]

print(f"Sampel untuk fit IForest : {len(X_fit):,}")
print(f"Sampel untuk kalibrasi    : {len(X_calib):,}")

# Inisialisasi dan latih Cascaded IForest
cascaded_iforest = CascadedIForest(
    n_estimators=config.iforest_n_estimators,
    n_jobs=config.iforest_n_jobs,
    random_state=config.seed,
    feature_mode=config.iforest_feature_mode,
)

print("Melatih Cascaded Isolation Forest...")
cascaded_iforest.fit(X_fit, sae_model)
cascaded_iforest.save(config.iforest_model_path)
print(f"Cascaded IForest berhasil dilatih dan disimpan ke {config.iforest_model_path}")

Mengambil sampel data train benign untuk melatih Cascaded IForest...
Total sampel train: 50,000
Sampel untuk fit IForest : 40,000
Sampel untuk kalibrasi    : 10,000
Melatih Cascaded Isolation Forest...
Cascaded IForest berhasil dilatih dan disimpan ke I:\skripsi\NIDS\notebooks\artifacts\cascaded_iforest.joblib


### 4. Normalisasi Skor Anomali & Kalibrasi Threshold pada Data Benign

Threshold anomali ($\tau$) ditentukan menggunakan data kalibrasi normal (*benign*) berdasarkan persentil (misal 95%), sehingga False Positive Rate (FPR) pada data normal dapat terkontrol (sekitar 5%).

In [5]:
hybrid_detector = HybridDetector(
    sae_model=sae_model,
    iforest_model=cascaded_iforest,
    fusion_strategy=config.fusion_strategy,
    alpha=config.alpha,
)

# Hitung batas min-max dari train set
sae_train_raw = sae_model.get_reconstruction_error(X_train_sample)
if_train_raw = cascaded_iforest.score_samples(X_train_sample, sae_model)
hybrid_detector.fit_bounds(sae_train_raw, if_train_raw)

# Kalibrasi threshold pada subset kalibrasi
sae_calib_raw = sae_model.get_reconstruction_error(X_calib)
if_calib_raw = cascaded_iforest.score_samples(X_calib, sae_model)
thresholds = hybrid_detector.calibrate_thresholds(
    sae_calib_raw,
    if_calib_raw,
    percentile=config.threshold_percentile,
)

print(f"=== Hasil Kalibrasi Threshold (Percentile = {config.threshold_percentile}%) ===")
print(f"Tau SAE               : {thresholds['tau_sae']:.6f}")
print(f"Tau Cascaded IForest  : {thresholds['tau_iforest']:.6f}")
print(f"Tau Max-Normalized    : {thresholds['tau_max_normalized']:.6f}")
print(f"Tau Weighted-Average  : {thresholds['tau_weighted_average']:.6f}")

=== Hasil Kalibrasi Threshold (Percentile = 95.0%) ===
Tau SAE               : 0.025457
Tau Cascaded IForest  : 0.625711
Tau Max-Normalized    : 1.179205
Tau Weighted-Average  : 0.324613


### 5. Evaluasi Komparatif pada Unseen Attack (Zero-Day Detection)

Kita memuat data uji yang terdiri dari lalu lintas normal yang belum pernah dilihat (*benign test*) dan serangan *unseen attack* (DDoS).

In [6]:
test_benign_files = sorted(config.test_benign_dir.glob("*.parquet"))
unseen_attack_files = sorted(config.unseen_attack_dir.glob("*.parquet"))

print("Memuat data evaluasi...")
X_test_b, y_test_b, att_b = load_evaluation_dataset(
    test_benign_files,
    feature_columns,
    mins,
    maxs,
    max_samples=config.eval_max_benign_samples,
    seed=config.seed,
)
X_test_a, y_test_a, att_a = load_evaluation_dataset(
    unseen_attack_files,
    feature_columns,
    mins,
    maxs,
    max_samples=config.eval_max_attack_samples,
    seed=config.seed + 1,
)

X_test = np.vstack([X_test_b, X_test_a])
y_test = np.concatenate([y_test_b, y_test_a])

print(f"Total Test Samples   : {len(X_test):,}")
print(f" - Normal (Benign)   : {len(X_test_b):,}")
print(f" - Unseen Attack     : {len(X_test_a):,}")

# Hitung skor anomali untuk semua sampel uji
scores_dict = hybrid_detector.compute_scores(X_test)

Memuat data evaluasi...
Total Test Samples   : 41,404
 - Normal (Benign)   : 16,404
 - Unseen Attack     : 25,000


In [7]:
# Evaluasi performa model tunggal vs strategi hybrid
results_df = evaluate_pipeline_models(y_test, scores_dict, thresholds)

print("\n" + "=" * 95)
print("TABEL PERBANDINGAN PERFORMA (FOKUS: PENURUNAN FNR & PENINGKATAN RECALL)")
print("=" * 95)
display(results_df.style.format({
    "TP": "{:,.0f}",
    "FP": "{:,.0f}",
    "TN": "{:,.0f}",
    "FN": "{:,.0f}",
    "FNR": "{:.4%}",
    "FPR": "{:.4%}",
    "Recall": "{:.4%}",
    "Precision": "{:.4%}",
    "F1": "{:.4f}",
    "ROC_AUC": "{:.4f}",
    "PR_AUC": "{:.4f}",
}))
print("=" * 95)


TABEL PERBANDINGAN PERFORMA (FOKUS: PENURUNAN FNR & PENINGKATAN RECALL)


,TP,FP,TN,FN,FNR,FPR,Recall,Precision,F1,ROC_AUC,PR_AUC
SAE Alone,"22,215",782,"15,622","2,785",11.1400%,4.7671%,88.8600%,96.5996%,0.9257,0.9727,0.9555
IForest Alone,"22,152",840,"15,564","2,848",11.3920%,5.1207%,88.6080%,96.3466%,0.9232,0.9666,0.9635
Hybrid (OR Fusion),"22,524","1,324","15,080","2,476",9.9040%,8.0712%,90.0960%,94.4482%,0.9222,0.9658,0.9518
Hybrid (Max Normalized),"22,204",760,"15,644","2,796",11.1840%,4.6330%,88.8160%,96.6905%,0.9259,0.9658,0.9518
Hybrid (Weighted Avg),"22,151",867,"15,537","2,849",11.3960%,5.2853%,88.6040%,96.2334%,0.9226,0.9709,0.9659


### 6. Visualisasi Kurva ROC, Precision-Recall, dan Confusion Matrix

Grafik ini siap dicantumkan dalam Bab Hasil dan Pembahasan Skripsi.

In [8]:
# 1. Plot Kurva ROC & PR
curves_path = config.results_dir / "hybrid_roc_pr_curves.png"
plot_roc_pr_curves(
    y_test,
    {
        "SAE Alone": scores_dict["sae_norm"],
        "Cascaded IForest": scores_dict["iforest_norm"],
        "Hybrid (Max-Norm)": scores_dict["hybrid_max_norm"],
    },
    save_path=curves_path,
)
print(f"Kurva ROC & PR tersimpan di: {curves_path}")

Kurva ROC & PR tersimpan di: I:\skripsi\NIDS\notebooks\results\hybrid_roc_pr_curves.png


In [9]:
# 2. Plot Confusion Matrices berdampingan
tau_sae = thresholds["tau_sae"]
tau_if = thresholds["tau_iforest"]

pred_sae = (scores_dict["sae_norm"] > tau_sae).astype(int)
pred_if = (scores_dict["iforest_norm"] > tau_if).astype(int)
pred_or = (pred_sae | pred_if).astype(int)

cm_path = config.results_dir / "hybrid_confusion_matrices.png"
plot_confusion_matrices(
    y_test,
    {
        "SAE Alone": pred_sae,
        "Cascaded IForest": pred_if,
        "Hybrid (OR-Rule)": pred_or,
    },
    save_path=cm_path,
)
print(f"Confusion matrix tersimpan di: {cm_path}")

Confusion matrix tersimpan di: I:\skripsi\NIDS\notebooks\results\hybrid_confusion_matrices.png
